In [1]:
!pip install langchain langchain-community huggingface_hub langsmith

In [3]:
!pip install -U 'langsmith[claude-agent-sdk]'

In [4]:
!pip install -U langchain-huggingface

In [5]:
!pip install transformers accelerate

In [6]:
from transformers import pipeline
from langchain_core.runnables import RunnableLambda

pipe = pipeline("text-generation", model="distilgpt2")

def local_llm(input):
    if hasattr(input, "to_string"):
        input = input.to_string()
    else:
        input = str(input)

    result = pipe(
        input,
        max_new_tokens=50,
        do_sample=False
    )[0]["generated_text"]

    return result.replace(input, "").strip()

llm = RunnableLambda(local_llm)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("Extract skills from: {resume}")

chain = prompt | llm


In [8]:
!pip install -U langchain langchain-core

In [9]:
from langchain_core.prompts import PromptTemplate

extraction_prompt = PromptTemplate.from_template("""
Extract only skills and experience.

Output format:
Skills: <comma separated>
Experience: <years>

Resume:
{resume}
""")

In [10]:
from langchain_core.output_parsers import StrOutputParser

extraction_chain = extraction_prompt | llm | StrOutputParser()

In [11]:
matching_prompt = PromptTemplate.from_template("""
Compare resume skills with job description.

Return:
Matched: <skills>
Missing: <skills>
Match Score: <number>

Resume Data:
{resume_data}

Job:
{job_description}
""")

matching_chain = matching_prompt | llm | StrOutputParser()

In [12]:
scoring_prompt = PromptTemplate.from_template("""
Give only a number between 0 and 100.

Data:
{match_data}
""")

scoring_chain = scoring_prompt | llm | StrOutputParser()

In [13]:
explanation_prompt = PromptTemplate.from_template("""
Explain the score in 2 lines.

Score: {score}
Data: {match_data}
""")

explanation_chain = explanation_prompt | llm | StrOutputParser()

In [24]:
import re

def extract_logic(resume):
    skills = []
    for s in ["python", "machine learning", "sql", "aws"]:
        if s in resume.lower():
            skills.append(s)

    exp_match = re.search(r"\d+", resume)
    experience = exp_match.group() + " years" if exp_match else "0 years"

    return f"Skills: {', '.join(skills)} | Experience: {experience}"

In [31]:
def match_logic(extracted, jd):
    jd_skills = ["python", "machine learning", "sql", "aws"]

    matched = [s for s in jd_skills if s in extracted.lower()]
    missing = [s for s in jd_skills if s not in extracted.lower()]

    score = int((len(matched)/len(jd_skills))*100)

    return {
        "matched": matched,
        "missing": missing,
        "score": score
    }

In [29]:
def scoring_logic(match_data):
    return match_data["score"]

In [30]:
def explanation_logic(score, match_data):
    return f"Candidate scored {score}/100 based on matched skills {match_data['matched']} and missing skills {match_data['missing']}."

In [32]:
def run_pipeline(resume, jd):
    extracted = extract_logic(resume)

    matched = match_logic(extracted, jd)

    score = scoring_logic(matched)

    explanation = explanation_logic(score, matched)

    return {
        "Extracted": extracted,
        "Matched": matched,
        "Score": score,
        "Explanation": explanation
    }

In [33]:
jd = "Python, Machine Learning, SQL, AWS"

resumes = [
    "Python ML SQL AWS 3 years",
    "Python SQL 1 year",
    "HTML CSS"
]

for r in resumes:
    print(run_pipeline(r, jd))

{'Extracted': 'Skills: python, sql, aws | Experience: 3 years', 'Matched': {'matched': ['python', 'sql', 'aws'], 'missing': ['machine learning'], 'score': 75}, 'Score': 75, 'Explanation': "Candidate scored 75/100 based on matched skills ['python', 'sql', 'aws'] and missing skills ['machine learning']."}
{'Extracted': 'Skills: python, sql | Experience: 1 years', 'Matched': {'matched': ['python', 'sql'], 'missing': ['machine learning', 'aws'], 'score': 50}, 'Score': 50, 'Explanation': "Candidate scored 50/100 based on matched skills ['python', 'sql'] and missing skills ['machine learning', 'aws']."}
{'Extracted': 'Skills:  | Experience: 0 years', 'Matched': {'matched': [], 'missing': ['python', 'machine learning', 'sql', 'aws'], 'score': 0}, 'Score': 0, 'Explanation': "Candidate scored 0/100 based on matched skills [] and missing skills ['python', 'machine learning', 'sql', 'aws']."}
